# NB_hospital_daily_summary

Builds `adb_caaresync.gold.hospital_daily_summary` — one row per hospital per calendar day with test volumes, patient counts, abnormal/critical rates, and day-over-day trend via LAG. Full rebuild via CREATE OR REPLACE TABLE.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caaresync.gold.hospital_daily_summary AS

WITH daily_agg AS (
  SELECT
    lr.hospital_id,
    lr.test_date                                                     AS summary_date,
    COUNT(*)                                                         AS total_tests,
    COUNT(DISTINCT lr.patient_id)                                    AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id)                                     AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)       AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)       AS critical_test_count
  FROM adb_caaresync.silver.lab_results lr
  GROUP BY lr.hospital_id, lr.test_date
),

enriched AS (
  SELECT
    da.hospital_id,
    h.hospital_name,
    da.summary_date,
    c.day_name,
    CAST(c.is_weekend AS BOOLEAN)                                    AS is_weekend,
    da.total_tests,
    da.unique_patients_tested,
    da.active_doctor_count,
    da.abnormal_test_count,
    da.critical_test_count,
    CAST(da.abnormal_test_count * 100.0 / da.total_tests AS DECIMAL(5,2))
                                                                     AS abnormal_rate_pct,
    CAST(da.critical_test_count * 100.0 / da.total_tests AS DECIMAL(5,2))
                                                                     AS critical_rate_pct
  FROM      daily_agg                          da
  LEFT JOIN adb_caaresync.silver.hospitals      h  ON da.hospital_id = h.hospital_id
  LEFT JOIN adb_caaresync.silver.calendar       c  ON c.date_key    = CAST(DATE_FORMAT(da.summary_date, 'yyyyMMdd') AS INT)
)

SELECT
  e.*,
  LAG(e.total_tests) OVER (PARTITION BY e.hospital_id ORDER BY e.summary_date)
                                                                     AS prev_day_total_tests,
  CAST(
    CASE
      WHEN LAG(e.total_tests) OVER (PARTITION BY e.hospital_id ORDER BY e.summary_date) IS NULL
        OR LAG(e.total_tests) OVER (PARTITION BY e.hospital_id ORDER BY e.summary_date) = 0
      THEN NULL
      ELSE (e.total_tests - LAG(e.total_tests) OVER (PARTITION BY e.hospital_id ORDER BY e.summary_date))
           * 100.0
           / LAG(e.total_tests) OVER (PARTITION BY e.hospital_id ORDER BY e.summary_date)
    END AS DECIMAL(6,2)
  )                                                                  AS test_volume_change_pct

FROM enriched e

In [0]:
record_count = spark.table("adb_caaresync.gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(record_count))